## Klasifikasi Topik Artikel Komdigi dengan KNN

Notebook ini menerapkan K-Nearest Neighbors pada artikel klarifikasi hoaks Komdigi. Teks dari `title`, `excerpt`, dan `body_text` diubah menjadi fitur TF-IDF; target klasifikasi adalah `topics`. `topics` merupakan kategori artikel, bukan label benar atau salahnya klaim. Nilai `k` dipilih melalui validasi silang pada data latih, kemudian model dievaluasi pada data uji.


In [ ]:
# Import library
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline

In [ ]:
# Cari dataset dari direktori kerja notebook hingga root proyek.
data_path = next(
    (
        parent / "data" / "komdigi_hoaks.csv"
        for parent in (Path.cwd(), *Path.cwd().parents)
        if (parent / "data" / "komdigi_hoaks.csv").is_file()
    ),
    None,
)
if data_path is None:
    raise FileNotFoundError("File data/komdigi_hoaks.csv tidak ditemukan dari direktori kerja saat ini.")

data = pd.read_csv(data_path)
required_columns = ["url", "title", "excerpt", "body_text", "topics"]
missing_columns = [column for column in required_columns if column not in data.columns]
if missing_columns:
    raise ValueError(f"Kolom wajib tidak ditemukan pada dataset: {missing_columns}")

# Hapus artikel duplikat, bersihkan nilai kosong, dan gabungkan kolom teks.
data = data.drop_duplicates(subset="url", keep="first").dropna(subset=["topics"]).copy()
text_columns = ["title", "excerpt", "body_text"]
data[text_columns] = data[text_columns].fillna("").astype(str)
data["article_text"] = (
    data[text_columns]
    .agg(" ".join, axis=1)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)
data = data.loc[data["article_text"].ne("")].reset_index(drop=True)

print(f"Lokasi dataset: {data_path.resolve()}")
print("Ukuran dataset setelah pembersihan:", data.shape)
display(data[["topics", "title", "excerpt"]].head())
data.info()

In [ ]:
# Ringkasan kategori target.
print("Jumlah artikel per topik:")
display(data["topics"].value_counts().rename_axis("topics").to_frame("jumlah"))
print("Jumlah nilai kosong pada teks dan target:")
print(data[["article_text", "topics"]].isnull().sum())

plt.figure(figsize=(10, 5))
class_order = data["topics"].value_counts().index
sns.countplot(data=data, y="topics", order=class_order, color="steelblue")
plt.title("Distribusi Topik Artikel Komdigi")
plt.xlabel("Jumlah artikel")
plt.ylabel("Topik")
plt.tight_layout()
plt.show()

In [ ]:
# Teks artikel menjadi fitur, sedangkan topics menjadi label kelas.
X = data["article_text"]
y = data["topics"]

# Pisahkan data 70:30 dengan menjaga proporsi setiap kelas.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y,
)
print("Jumlah data latih:", len(X_train))
print("Jumlah data uji:", len(X_test))

In [ ]:
# TF-IDF mengubah artikel menjadi vektor teks sparse.
# Jarak cosine sesuai untuk membandingkan kemiripan dokumen.
knn_pipeline = Pipeline(
    steps=[
        (
            "tfidf",
            TfidfVectorizer(
                max_features=10000,
                min_df=2,
                ngram_range=(1, 2),
                sublinear_tf=True,
            ),
        ),
        (
            "knn",
            KNeighborsClassifier(metric="cosine", algorithm="brute"),
        ),
    ]
)

# Pilih jumlah tetangga menggunakan validasi silang hanya pada data latih.
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
param_grid = {"knn__n_neighbors": list(range(1, 11))}
knn_search = GridSearchCV(
    knn_pipeline,
    param_grid=param_grid,
    scoring="f1_macro",
    cv=cv,
    n_jobs=-1,
    refit=True,
)
knn_search.fit(X_train, y_train)
knn = knn_search.best_estimator_
print("Nilai k terbaik:", knn_search.best_params_["knn__n_neighbors"])
print("Macro F1 validasi silang:", round(knn_search.best_score_, 4))

In [ ]:
# Evaluasi sekali pada data uji yang tidak digunakan untuk memilih nilai k.
y_pred = knn.predict(X_test)
labels = sorted(y.unique())
print("Akurasi data uji:", round(accuracy_score(y_test, y_pred), 4))
print("Macro F1-score data uji:", round(f1_score(y_test, y_pred, average="macro", zero_division=0), 4))
print("\nLaporan klasifikasi:")
print(
    classification_report(
        y_test,
        y_pred,
        labels=labels,
        target_names=labels,
        zero_division=0,
    )
)

In [ ]:
# Tampilkan performa validasi silang untuk setiap nilai k.
cv_results = pd.DataFrame(knn_search.cv_results_)
cv_summary = pd.DataFrame({
    "k": cv_results["param_knn__n_neighbors"].astype(int),
    "Macro F1 validasi silang": cv_results["mean_test_score"],
    "Standar deviasi": cv_results["std_test_score"],
})
display(cv_summary.sort_values("k", ignore_index=True))

plt.figure(figsize=(8, 5))
plt.errorbar(
    cv_summary["k"],
    cv_summary["Macro F1 validasi silang"],
    yerr=cv_summary["Standar deviasi"],
    marker="o",
    capsize=4,
)
plt.title("Pemilihan k KNN dengan Validasi Silang")
plt.xlabel("Jumlah tetangga (k)")
plt.ylabel("Macro F1 validasi silang")
plt.xticks(cv_summary["k"])
plt.tight_layout()
plt.show()

cm = confusion_matrix(y_test, y_pred, labels=labels)
plt.figure(figsize=(9, 7))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=labels,
    yticklabels=labels,
)
plt.title("Confusion Matrix - KNN Topik Komdigi")
plt.xlabel("Prediksi")
plt.ylabel("Label sebenarnya")
plt.xticks(rotation=35, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Tinjau prediksi untuk beberapa artikel dalam data uji.
hasil_prediksi = data.loc[X_test.index, ["title", "topics"]].rename(
    columns={"topics": "topics_asli"}
).copy()
hasil_prediksi["topics_prediksi"] = y_pred
display(hasil_prediksi.head(10))